# afriseg 01: prepare data and calibrate

**Before running:**
1. In Settings, turn **Internet on**. The accelerator can stay off, because this notebook is CPU-only.
2. Use **Add Input** to attach two datasets:
   - `dschettler8845/brats-2021-task1`
   - your private BraTS-Africa dataset, made from `afriseg-africa-npz.zip`

Attach **only** these two inputs. Remove any other attached notebook outputs, because the
BraTS 2021 search would pick up their files too.

Run it in the background: **Save Version**, then **Save & Run All (Commit)**. That keeps running for
up to 12 hours after you close the browser; an interactive session stops when you leave. It takes
roughly 2 to 3 hours.
3. When it finishes, click **Save Version** and choose "Save & Run All" or "Quick Save". The
   notebook's output (`/kaggle/working`) then becomes an input for the training notebook.


In [ ]:
# 1. Get the latest code
!rm -rf /kaggle/working/afriseg && git clone -q https://github.com/AJmoe/afriseg.git /kaggle/working/afriseg
!pip install -q -e /kaggle/working/afriseg
!cd /kaggle/working/afriseg && git log --oneline -1

In [ ]:
# 2. Find the attached datasets (Kaggle mounts them under /kaggle/input, sometimes in /kaggle/input/datasets/<owner>/)
# BraTS-Africa is accepted in any of three forms:
#   a) the prepared folder (africa_manifest.csv + africa/*.npz)
#   b) the prepared zip that Kaggle did not unpack (afriseg-africa-npz.zip)
#   c) the original TCIA download (BraTS-SSA-*.nii.gz files) -> preprocessed here
import glob, os, subprocess, json, collections, zipfile
print('Attached inputs:')
for d in sorted(glob.glob('/kaggle/input/*')) + sorted(glob.glob('/kaggle/input/datasets/*/*')):
    print('  ', d)

africa_manifest = glob.glob('/kaggle/input/**/africa_manifest.csv', recursive=True)
if not africa_manifest:
    zips = glob.glob('/kaggle/input/**/afriseg-africa-npz*.zip', recursive=True)
    raw = glob.glob('/kaggle/input/**/BraTS-SSA-*-t2f.nii.gz', recursive=True)
    if zips:
        print('unzipping', zips[0])
        zipfile.ZipFile(zips[0]).extractall('/kaggle/working/africa_npz')
    elif raw:
        print('found', len(raw), 'original BraTS-Africa cases; preprocessing the 95 gliomas')
        root = os.path.commonpath([os.path.dirname(f) for f in raw])
        subprocess.run(['python', '-m', 'afriseg.data', 'preprocess', '--root', root,
                        '--out', '/kaggle/working/africa_npz', '--dataset', 'africa', '--workers', '4',
                        '--ids', '/kaggle/working/afriseg/splits/africa_glioma_ids.txt'], check=True)
    africa_manifest = glob.glob('/kaggle/working/africa_npz/**/africa_manifest.csv', recursive=True)
assert africa_manifest, 'BraTS-Africa not found: attach it with Add Input (see the list above)'
AFRICA = os.path.dirname(africa_manifest[0])
n_af = len(glob.glob(AFRICA + '/africa/*.npz'))
print('BraTS-Africa:', AFRICA, n_af, 'cases')
assert n_af == 95, f'Expected 95 glioma cases, found {n_af}'

nii = glob.glob('/kaggle/input/**/BraTS2021_*_flair.nii.gz', recursive=True)
tars = sorted(glob.glob('/kaggle/input/**/*.tar', recursive=True), key=os.path.getsize)
print('BraTS 2021: nifti files found:', len(nii), '| tar files:', tars)
assert nii or tars, 'BraTS 2021 dataset not attached'


In [ ]:
# 3. Unpack BraTS 2021 if it is only available as .tar archives (about 10 minutes)
if len(nii) >= 1000:
    BRATS_ROOT = os.path.commonpath(nii[:50] + nii[-50:])
else:
    BRATS_ROOT = '/tmp/brats2021'
    os.makedirs(BRATS_ROOT, exist_ok=True)
    big = tars[-1]  # the training archive is by far the largest file
    print('extracting', big)
    subprocess.run(['tar', '-xf', big, '-C', BRATS_ROOT], check=True)
n = len(glob.glob(BRATS_ROOT + '/**/*_flair.nii.gz', recursive=True))
print('BraTS 2021 root:', BRATS_ROOT, '| cases:', n)
assert n > 1000, 'Expected about 1251 BraTS 2021 cases'

In [ ]:
# 4. Preprocess BraTS 2021 with 4 CPU processes (about 30 to 60 minutes)
!rm -rf /kaggle/working/npz
!cd /kaggle/working && python -m afriseg.data preprocess --root {BRATS_ROOT} --out /kaggle/working/npz --dataset brats2021 --workers 4 | grep -v '^\[' | tail -5
import pandas as pd
m21 = pd.read_csv('/kaggle/working/npz/brats2021_manifest.csv')
print(len(m21), 'cases | label schemes:', m21.scheme.value_counts().to_dict(), '| first ids:', sorted(m21.id)[:2])
!du -sh /kaggle/working/npz
assert 1200 <= len(m21) <= 1300, f'Expected about 1251 BraTS 2021 training cases, got {len(m21)}: wrong input attached?'


In [ ]:
# 5. Frozen splits: BraTS 2021 holdout 80/10/10; copy the BraTS-Africa 5-fold split from the repo
!mkdir -p /kaggle/working/splits /kaggle/working/configs /kaggle/working/quality
!python -m afriseg.data split --manifest /kaggle/working/npz/brats2021_manifest.csv --mode holdout --out /kaggle/working/splits/brats2021.json
!cp /kaggle/working/afriseg/splits/africa.json /kaggle/working/splits/africa.json
import json, collections
s = json.load(open('/kaggle/working/splits/brats2021.json'))['split']
print(collections.Counter(s.values()))

In [ ]:
# 6. Calibration subset: images of BraTS-Africa folds 0 and 1 only (no labels are used)
af = json.load(open('/kaggle/working/splits/africa.json'))['split']
calib = sorted(k for k, v in af.items() if v in (0, 1))
json.dump(calib, open('/kaggle/working/splits/africa_calib_ids.json', 'w'))
print(len(calib), 'calibration cases')
qual = os.path.join(AFRICA, 'africa_quality.csv')
if not os.path.exists(qual):
    qual = '/kaggle/working/quality/africa.csv'
    subprocess.run(['python', '-m', 'afriseg.quality', '--manifest', africa_manifest[0], '--out', qual], check=True)
print('target quality features:', qual)

In [ ]:
# 7. Label-free calibration search (about 1 to 2 hours on CPU)
!python -m afriseg.calibrate --source-manifest /kaggle/working/npz/brats2021_manifest.csv     --target-features {qual} --target-ids /kaggle/working/splits/africa_calib_ids.json     --n-candidates 30 --n-source 20 --repeats 2 --out /kaggle/working/configs/aug15t_calibrated.json
!cat /kaggle/working/configs/aug15t_calibrated.json

## Done
Look at the last line of step 7. The calibrated distance should be clearly below the
"no augmentation" distance; if it is not, report that before training.

Download `configs/aug15t_calibrated.json` and `configs/aug15t_calibrated.search.json` from the
Output panel and add them to the GitHub repo. Then **Save Version** so the preprocessed data can be
attached to the training notebook.